# 02 — Features

**Filter**
- `is_contract` — has bytecode (cluster EOAs separately)

**Tier 1 — Clustering features**
- `account_age_days` — days since first tx
- `tx_count` — total txs sent
- `active_days` — distinct days with ≥1 tx
- `tx_per_active_day` — intensity (tx_count / active_days)
- `median_tx_interval_sec` — median seconds between txs
- `tx_interval_std_sec` — std dev of inter-tx intervals
- `active_hour_entropy` — Shannon entropy of hour-of-day distribution
- `unique_peers` — distinct counterparty addresses
- `counterparty_concentration_top3` — % of txs to top-3 counterparties
- `contract_call_ratio` — % outbound txs to contracts
- `cex_interaction_ratio` — % txs touching known CEX addresses
- `volume_asymmetry` — (in − out) / (in + out + ε)
- `stablecoin_volume_ratio` — share of ERC-20 volume in USDC/USDT/DAI
- `nft_activity_ratio` — % txs that are ERC-721/1155 transfers or marketplace calls
- `median_hold_time_days` — median days between token in and token out

## Setup

In [5]:
import pandas as pd, numpy as np

BigQuery names their shards in 12 digits.

1. For blocks, we merge into single file
2. For transactions and logs, we merge into monthly

In [ ]:
import duckdb

# Step 1: Combine all block files into a single Parquet file
duckdb.sql("""
    COPY (
        SELECT * FROM '../data/bronze/blocks/blocks-*.parquet'
        ORDER BY block_number
    ) TO '../data/bronze/blocks.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

# Step 2: Analyze the combined Parquet file
duckdb.sql("""
    SELECT 
        COUNT(*) AS rows,
        MIN(block_number) AS min_block,
        MAX(block_number) AS max_block,
        COUNT(DISTINCT block_number) AS unique_blocks
    FROM '../data/bronze/blocks.parquet'
""").show()

In [1]:
import duckdb

duckdb.sql("SET threads = 10")
duckdb.sql("SET memory_limit = '12GB'")
duckdb.sql("SET preserve_insertion_order = false")

In [3]:
duckdb.sql("""
    COPY (
        SELECT *,
               STRFTIME(block_timestamp, '%Y-%m') AS year_month
        FROM '../data/bronze/transactions/transactions-*.parquet'
    ) TO '../data/bronze/transactions/'
    (FORMAT PARQUET, PARTITION_BY (year_month), COMPRESSION ZSTD, OVERWRITE_OR_IGNORE)
""")

IOException: IO Error: Cannot open file "../data/bronze/transactions/transactions-000000000931.parquet": No such file or directory

In [ ]:


# ── load bronze ──
bronze_blocks = pd.read_parquet("../data/bronze/blocks/blocks.parquet")
# bronze_tx     = pd.read_parquet("../data/bronze/transactions.parquet")
# bronze_logs   = pd.read_parquet("../data/bronze/logs.parquet")



FileNotFoundError: [Errno 2] No such file or directory: '../data/bronze/blocks/blocks.parquet'

In [ ]:
# ── silver: blocks ──
def silver_blocks(df):
    out = df.copy()

    # extract useful time parts
    out["date"]  = out["timestamp"].dt.date
    out["hour"]  = out["timestamp"].dt.hour

    # drop low-value fields for analysis
    out = out.drop(columns=["logs_bloom", "sha3_uncles", "extra_data",
                             "state_root", "receipts_root", "transactions_root"], errors="ignore")

    # flag post-merge (difficulty = 0)
    out["is_post_merge"] = out["difficulty"] == 0

    return out



In [ ]:
# ── silver: transactions ──
def silver_transactions(df):
    out = df.copy()

    # split receipt_* cols into separate df
    receipt_cols = [c for c in df.columns if c.startswith("receipt_")]
    base_cols    = [c for c in df.columns if not c.startswith("receipt_")]

    silver_tx  = out[base_cols].copy()
    silver_rec = out[["hash", "block_number", "block_timestamp"] + receipt_cols].copy()

    # rename receipt cols — drop prefix
    silver_rec.columns = [c.replace("receipt_", "") if c.startswith("receipt_") else c
                          for c in silver_rec.columns]

    # wei → ether
    silver_tx["value_eth"] = silver_tx["value"].astype(float) / 1e18

    # flag contract deployments (no to_address)
    silver_tx["is_deploy"] = silver_tx["to_address"].isna()

    # extract time parts
    silver_tx["date"] = silver_tx["block_timestamp"].dt.date
    silver_tx["hour"] = silver_tx["block_timestamp"].dt.hour

    return silver_tx, silver_rec

# ── silver: logs ──
def silver_logs(df):
    out = df.copy()

    # explode topics into separate columns (topic0 = event signature)
    out["topic0"] = out["topics"].apply(lambda t: t[0] if isinstance(t, list) and len(t) > 0 else None)
    out["topic1"] = out["topics"].apply(lambda t: t[1] if isinstance(t, list) and len(t) > 1 else None)
    out["topic2"] = out["topics"].apply(lambda t: t[2] if isinstance(t, list) and len(t) > 2 else None)
    out["topic3"] = out["topics"].apply(lambda t: t[3] if isinstance(t, list) and len(t) > 3 else None)

    # drop raw topics col
    out = out.drop(columns=["topics"])

    # extract time parts
    out["date"] = out["block_timestamp"].dt.date
    out["hour"] = out["block_timestamp"].dt.hour

    return out

# ── run ──
silver_blocks_df             = silver_blocks(bronze_blocks)
silver_tx_df, silver_rec_df  = silver_transactions(bronze_tx)
silver_logs_df               = silver_logs(bronze_logs)

# ── save silver ──
silver_blocks_df.to_parquet("silver_blocks.parquet",       index=False)
silver_tx_df.to_parquet("silver_transactions.parquet",     index=False)
silver_rec_df.to_parquet("silver_receipts.parquet",        index=False)
silver_logs_df.to_parquet("silver_logs.parquet",           index=False)

print("silver_blocks:      ", silver_blocks_df.shape)
print("silver_transactions:", silver_tx_df.shape)
print("silver_receipts:    ", silver_rec_df.shape)
print("silver_logs:        ", silver_logs_df.shape)

In [13]:
# Tool Imports
import sys, pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path

# Project imports
# sys.path.insert(0, str(Path.cwd().parent)) # inserts github project parent to PATH
from src.features import build_features, scale_features
from src.models import fit_kmeans, cluster_profiles

# pandas display options: max_columns=50, max_colwidth=20
pd.options.display.max_columns = 50
pd.options.display.max_colwidth = 20
PROCESSED = Path("../data/processed")

## 1. Load features

Build the feature matrix if it doesn't exist yet, otherwise reload from disk.
This pattern keeps notebook re-runs fast.

In [ ]:
# If wallet_features.parquet exists -> pd.read_parquet
# Else -> build_features() and save
# Print shape and column list

features = None  # TODO

In [ ]:
# Quick sanity glance
# - features.head()
# - features.dtypes  (no unexpected object columns)
# - features.isna().sum()  (should be all zeros)

## 2. Raw feature distributions

Looking for: heavy tails (need log), zero-inflation (suspicious), bimodality (potential cluster signal).

In [ ]:
# Define the feature columns you'll inspect (mirror FEATURE_COLS from src/features/scaling.py)
FEATURE_COLS = [
    # TODO: paste from src/features/scaling.py
]

In [ ]:
# Histogram grid of raw features
# - 4x4 subplots (or similar), one feature per axis
# - bins=50
# - log y-scale if a tail dominates
#
# What to look for:
# - single spike at zero -> dead feature
# - long right tail -> log-transform needed
# - visible bimodality -> potential cluster axis

In [ ]:
# Summary statistics
# features[FEATURE_COLS].describe(percentiles=[.01, .25, .5, .75, .9, .99])
#
# What to look for:
# - 99th percentile >> median -> heavy tail
# - min and median both 0 -> mostly empty

In [ ]:
# Zero-inflation check
# For each feature: (features[col] == 0).mean()
# Print as a sorted Series, descending
#
# Decision: any feature with >95% zeros is probably dead weight.
# Note candidates to drop.

## 3. Scaling

Log1p the heavy-tailed columns, then StandardScaler the whole matrix.
See `src/features/scaling.py` for the column lists.

In [ ]:
# Run scaling
# scaled = scale_features(features)
# wallets = scaled["wallet"]
# X = scaled.drop(columns="wallet")

In [ ]:
# Confirm scaling worked
# X.describe().loc[["mean", "std"]].round(3)
#
# Means ~ 0, stds ~ 1. If not, stop and debug upstream.

In [ ]:
# Histogram grid of scaled features (same layout as raw histograms)
#
# What to look for:
# - bell-ish shapes -> good
# - spike + outliers -> essentially binary, weak feature
# - long tail even after log -> consider clipping at 99th percentile

## 4. Correlation analysis

K-Means has no correlation regularization. Two near-duplicate features double-weight that axis.
Goal: identify and drop redundant features.

In [ ]:
# Correlation heatmap
# - X.corr()
# - sns.heatmap with annot=True, fmt=".2f", cmap="RdBu_r", center=0
# - figsize ~ (12, 10) so numbers are readable
#
# What to look for: any pair with |corr| > 0.9.
# Likely culprits: tx_count <-> active_days,
#                  tokens_sent_count <-> unique_tokens_sent,
#                  unique_recipients <-> tx_count

In [ ]:
# Programmatic correlation flagging
# - Take the upper triangle of X.corr().abs()
# - Stack to long form, filter > 0.9, sort descending
# - Print offending pairs
#
# Decision: for each high-corr pair, pick which to keep (usually the more interpretable one).

### Feature pruning decisions

_Write 3-5 bullets here documenting what you're dropping and why. Example:_

- Dropping `active_days` (corr 0.94 with `tx_count`, less interpretable)
- Dropping `unique_tokens_sent` (corr 0.92 with `tokens_sent_count`)
- Keeping `failed_tx_ratio` despite low variance — strong bot signal
- ...

In [ ]:
# Apply pruning (if you decided to drop features above)
FINAL_FEATURES = [
    # TODO: final list after pruning
]

# X = X[FINAL_FEATURES]

## 5. Dimensionality preview (PCA)

30-second answer to "is there structure here?"
If 2-3 PCs explain >70% variance, K-Means will likely find clean clusters.
If you need 10+ PCs, expect noisier clusters.

In [ ]:
# PCA scree plot
# - PCA(n_components=len(FINAL_FEATURES)).fit(X)
# - Plot cumulative explained variance vs # components
# - Add horizontal lines at 0.7, 0.8, 0.9
#
# Read off: how many PCs to hit 80%?

In [ ]:
# PCA 2D scatter
# - Project X to 2 components
# - Scatter alpha=0.3, small markers
# - Color by one feature you expect to dominate (e.g., log(tx_count) or contract_call_ratio)
#
# What to look for: clumps, arms, density variations.
# Uniform blob = clustering will struggle.
# Fingers/islands = good signs.

## 6. Clusterability preview

Quick K-Means at k=6 to confirm features carry signal.
**Not the final clustering** — that's `03_clustering.ipynb`. This is a smoke test.

In [ ]:
# Quick K-Means at k=6
# from src.models import fit_kmeans
# model, labels = fit_kmeans(X, k=6)
#
# Print cluster sizes: pd.Series(labels).value_counts().sort_index()
#
# What to look for: sizes in 100s to 1000s, no single cluster with 95%+ of wallets.

In [ ]:
# Preview cluster profiles
# from src.models import cluster_profiles
# profiles = cluster_profiles(scaled, labels)  # pass scaled DF, including wallet column
#
# Display as heatmap: center=0, cmap="RdBu_r", annot=True, fmt=".1f"
#
# What to look for:
# - Each cluster should have 2-3 features clearly +/- vs global mean
# - If every cluster looks the same -> features don't discriminate, go back and improve
# - If you can already squint and see archetypes (high-activity, whale-like, etc.) -> ready for 03

## 7. Final feature set lock-in

_Fill in:_

- **Wallets in final matrix:** N
- **Features kept:** ...
- **Features dropped (and why):** ...
- **Clusterability assessment:** (one sentence based on sections 5-6)

In [ ]:
# Save final scaled matrix
# - Reattach wallet column to X (if you pruned, the columns are now FINAL_FEATURES)
# - Write to PROCESSED / "wallet_features_scaled.parquet"
# - This overwrites the version produced by scale_features() — intentional.

## Handoff to 03_clustering.ipynb

_Write 2-3 sentences for future-you:_

- Path to scaled features: `data/processed/wallet_features_scaled.parquet`
- Expected number of clusters based on PCA: ...
- First Etherscan-validation candidates: wallets with extreme `tx_count` and `failed_tx_ratio`